# 로짓 덤프 — 3모델 × {test, val}

- 산출물: `logits_{tag}_{split}.npy`(`[N, 188]` fp32) + `doc_ids_{split}.json`
- 로짓 행 순서 = `doc_ids_{split}.json` 순서 = 토큰화 데이터셋 행 순서

In [1]:
!wget "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

--2026-07-23 01:48:23--  https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1005888967/c7881447-d72a-40ab-af45-52bfe18615f4?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-23T02%3A26%3A35Z&rscd=attachment%3B+filename%3Dflash_attn-2.8.3%2Bcu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-23T01%3A25%3A59Z&ske=2026-07-23T02%3A26%3A35Z&sks=b&skv=2018-11-09&sig=UCpS21QcBg6Lvt5IHiwQDlUHFJghNQqLvsl1uAt5XO0%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHM

In [2]:
import os
import gc
import json
import random
from pathlib import Path

import numpy as np
from numpy.typing import NDArray
from dotenv import load_dotenv

load_dotenv()

# 로컬
# ROOT = Path(os.environ["DATA_ROOT"])
# os.environ["HF_HOME"] = str(ROOT / ".hf_cache")

# 코랩
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_HOME"] = "/content/.hf_cache"

import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "splits": ["test", "val"],
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],    # 기록용
    "hf_cache": "/content/.hf_cache",
    "out_path": "/content/drive/MyDrive/patent_disc/output/",           # 로짓·SSOT metrics 모두 Drive output/ 직결
}

# 모델별 사양
MODELS = [
    {
        "tag": "modernbert-patent-len512-zlpr",
        "arch": "modernbert",
        "ckpt": "ingyoun/A.X-patent-len512-zlpr",
        "tokenizer": "ingyoun/A.X-patent-len512-zlpr",
        "tok_rev": "7f2232d48d8c72eadd988d40bdfe9a2e0a62fc79",
        "token_ds": "ingyoun/patent-clean-text-modernbert-tokenized",
        "max_len": 512,
        "batch_size": 512,
    },
    {
        "tag": "modernbert-patent-len512-asl",
        "arch": "modernbert",
        "ckpt": "ingyoun/A.X-patent-len512-ASL",
        "tokenizer": "ingyoun/A.X-patent-len512-ASL",
        "tok_rev": "8aced67224c3c4a395d16a1e089a31ac3faf0aa4",
        "token_ds": "ingyoun/patent-clean-text-modernbert-tokenized",
        "max_len": 512,
        "batch_size": 512,
    },
    {
        "tag": "modernbert-patent-len512-bce",
        "arch": "modernbert",
        "ckpt": "ingyoun/A.X-patent-len512-bce",
        "tokenizer": "ingyoun/A.X-patent-len512-bce",
        "tok_rev": "efddfdad0de28d1609b8356122d0d73aa9fe48d7",
        "token_ds": "ingyoun/patent-clean-text-modernbert-tokenized",
        "max_len": 512,
        "batch_size": 512,
    },
]

In [5]:
# fix seed
random.seed(config["seed"])
np.random.seed(config["seed"])
torch.manual_seed(config["seed"])
torch.cuda.manual_seed_all(config["seed"])

In [6]:
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

NVIDIA L4


## 추론

In [7]:
class EvalCollator:
    """동적 패딩"""
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, feats):
        enc = [
            {"input_ids": f["input_ids"],
            "attention_mask": f["attention_mask"]}
            for f in feats
        ]
        return self.tok.pad(enc, padding=True, return_tensors="pt")


class LogitsRunner:
    """모델 1개 → split별 logits를 추론·캐시"""
    def __init__(self, model, tokenizer, cache_dir, tag: str, arch: str, batch_size: int):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = model.to(self.device).eval()
        self.collate = EvalCollator(tokenizer)
        self.cache_dir = Path(cache_dir)
        self.tag, self.arch, self.batch_size = tag, arch, batch_size

    @torch.no_grad
    def _infer(self, ds) -> NDArray:
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)
        dtype = torch.bfloat16 if self.arch == "modernbert" else torch.float16
        chunks = []
        for enc in tqdm(loader, desc=self.tag):
            enc = {k: v.to(self.device) for k, v in enc.items()}
            if self.device == "cuda":
                with torch.autocast("cuda", dtype=dtype):
                    logits = self.model(**enc).logits
            else:
                logits = self.model(**enc).logits
            chunks.append(logits.float().cpu().numpy())
        return np.concatenate(chunks, axis=0)

    def get(self, ds, split) -> NDArray:
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        fp = self.cache_dir / f"logits_{self.tag}_{split}.npy"
        if fp.exists():
            print(f"[skip] {fp.name}")
            return np.load(fp)
        arr = self._infer(ds)
        np.save(fp, arr)
        print(f"[save] {fp.name} {arr.shape}")
        return arr

## Orchestrator

In [8]:
def save_doc_ids(out_path, split, doc_ids):
    """split별 document_id 축을 1회 저장하고, 이후 모델은 같은 순서인지 검증한다.
    로짓 행 순서 = 세 모델의 토큰화 데이터셋은 같은 clean-text에서 같은 순서로 파생
    """
    fp = Path(out_path) / f"doc_ids_{split}.json"
    fp.parent.mkdir(parents=True, exist_ok=True)
    if fp.exists():
        assert json.loads(fp.read_text()) == list(doc_ids), f"{split}: document_id 순서가 모델 간 불일치"
        return
    fp.write_text(json.dumps(list(doc_ids)))


class LogitDumpHarness:
    """모델 사양 1개 → 지정 split들의 로짓 덤프."""
    def __init__(self, spec: dict, config: dict):
        self.spec, self.cfg = spec, config

    @staticmethod
    def _truncate(ds, tokenizer, max_len):
        """ max_len으로 절단 — 선두 <s> 유지 + 꼬리를 eos로 마감"""
        eos_id = tokenizer.eos_token_id
        def _fn(batch):
            ids, masks = [], []
            for x, m in zip(batch["input_ids"], batch["attention_mask"]):
                if len(x) > max_len:
                    x = x[: max_len - 1] + [eos_id]
                    m = m[:max_len]
                ids.append(x)
                masks.append(m)
            return {"input_ids": ids, "attention_mask": masks}
        return ds.map(_fn, batched=True)

    def _load_tokenizer(self):
        kw = {"revision": self.spec["tok_rev"]}
        if self.spec["arch"] == "kobert":
            kw["trust_remote_code"] = True
        return AutoTokenizer.from_pretrained(self.spec["tokenizer"], **kw)

    def _load_model(self):
        if self.spec["arch"] == "modernbert":
            return AutoModelForSequenceClassification.from_pretrained(
                pretrained_model_name_or_path=self.spec["ckpt"],
                dtype=torch.float32,                   # fp32 로드 + autocast bf16 (평가 프로토콜)
                attn_implementation="flash_attention_2",
            )
        return AutoModelForSequenceClassification.from_pretrained(
            pretrained_model_name_or_path=self.spec["ckpt"]
        )

    def run(self, splits):
        cfg, spec = self.cfg, self.spec
        tokenizer = self._load_tokenizer()
        model = self._load_model()
        runner = LogitsRunner(
            model=model, tokenizer=tokenizer, cache_dir=cfg["out_path"],
            tag=spec["tag"], arch=spec["arch"], batch_size=spec["batch_size"],
        )
        for split in splits:
            ds = load_dataset(spec["token_ds"], cache_dir=cfg["hf_cache"], split=split)
            save_doc_ids(cfg["out_path"], split, ds["document_id"])
            n = len(ds)
            if spec["max_len"] is not None:            # KoBERT 데이터셋은 이미 512로 잘려 있다
                ds = self._truncate(ds, tokenizer, spec["max_len"])
            logits = runner.get(ds, split)
            assert logits.shape == (n, cfg["num_labels"]), f"{spec['tag']}/{split}: {logits.shape} != ({n}, {cfg['num_labels']})"

        del runner, model
        gc.collect()
        torch.cuda.empty_cache()

## 실행

In [9]:
for spec in MODELS:
    print(f"\n=== {spec['tag']} ===")
    LogitDumpHarness(spec, config).run(config["splits"])


=== modernbert-patent-len512-zlpr ===


config.json:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  598MB            

...

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

modernbert-patent-len512-bce:   0%|          | 0/23 [00:00<?, ?it/s]

[save] logits_modernbert-patent-len512-bce_test.npy (11271, 188)


modernbert-patent-len512-bce:   0%|          | 0/22 [00:00<?, ?it/s]

[save] logits_modernbert-patent-len512-bce_val.npy (11162, 188)


## 검증

test 로짓으로 headline 지표를 재계산해 `output/{tag}_test_metrics.json`(trainer.evaluate 산출, flat `test_*` 키)과 4자리 일치를 확인. SSOT 파일은 `out_path`(= `output/`)에 둔다.

In [10]:
def verify_test(spec, config) -> bool:
    logits = np.load(Path(config["out_path"]) / f"logits_{spec['tag']}_test.npy")
    ds = load_dataset(spec["token_ds"], cache_dir=config["hf_cache"], split="test")
    Y = np.asarray(ds["labels"], dtype=int)
    pred = (logits >= 0.0).astype(int)   # ZLPR native 임계(logit≥0). sigmoid≥0.5와 동치이나 확률 프레이밍 없이 직접 표기
    got = {
        "micro": f1_score(Y, pred, average="micro", zero_division=0),
        "macro": f1_score(Y, pred, average="macro", zero_division=0),
        "sample": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),
    }

    # SSOT = trainer.evaluate가 낸 {tag}_test_metrics.json (flat test_* 키)
    fp = Path(config["out_path"]) / f"{spec['tag']}_test_metrics.json"
    if not fp.exists():
        print(f"  [warn] SSOT 없음: {fp} — 로컬 output/과 대조할 것")
        print("  " + "  ".join(f"{k}={v:.4f}" for k, v in got.items()))
        return False

    ref = json.loads(fp.read_text(encoding="utf-8"))
    want = {
        "micro": ref["test_micro_f1"],
        "macro": ref["test_macro_f1"],
        "sample": ref["test_sample_f1"],
        "empty_rate": ref["test_empty_rate"],
        "anchor_weighted_f1": ref["test_anchor_weighted_f1"],
    }
    ok = True
    for k, v in got.items():
        hit = round(v, 4) == round(want[k], 4)
        ok &= hit
        print(f"  {'OK  ' if hit else 'FAIL'} {k:20s} got={v:.4f} ssot={want[k]:.4f}")
    return ok


for spec in MODELS:
    print(f"\n=== {spec['tag']} ===")
    verify_test(spec, config)


=== modernbert-patent-len512-zlpr ===
  OK   micro                got=0.8493 ssot=0.8493
  OK   macro                got=0.8462 ssot=0.8462
  OK   sample               got=0.8662 ssot=0.8662
  OK   empty_rate           got=0.0096 ssot=0.0096
  OK   anchor_weighted_f1   got=0.8122 ssot=0.8122

=== modernbert-patent-len512-asl ===
  OK   micro                got=0.8362 ssot=0.8362
  OK   macro                got=0.8366 ssot=0.8366
  OK   sample               got=0.8646 ssot=0.8646
  OK   empty_rate           got=0.0043 ssot=0.0043
  OK   anchor_weighted_f1   got=0.8148 ssot=0.8148

=== modernbert-patent-len512-bce ===
  OK   micro                got=0.8538 ssot=0.8538
  OK   macro                got=0.8508 ssot=0.8508
  OK   sample               got=0.8689 ssot=0.8689
  OK   empty_rate           got=0.0106 ssot=0.0106
  OK   anchor_weighted_f1   got=0.8133 ssot=0.8133


In [11]:
for fp in sorted(Path(config["out_path"]).glob("*")):
    print(f"{fp.name:44s} {fp.stat().st_size / 1e6:8.1f} MB")

doc_ids_test.json                                 0.2 MB
doc_ids_val.json                                  0.2 MB
logits_kobert-patent-baseline_len512_test.npy      8.5 MB
logits_kobert-patent-baseline_len512_val.npy      8.4 MB
logits_modernbert-patent-len512-asl_test.npy      8.5 MB
logits_modernbert-patent-len512-asl_val.npy       8.4 MB
logits_modernbert-patent-len512-bce_test.npy      8.5 MB
logits_modernbert-patent-len512-bce_val.npy       8.4 MB
logits_modernbert-patent-len512-zlpr_test.npy      8.5 MB
logits_modernbert-patent-len512-zlpr_val.npy      8.4 MB
logits_modernbert-patent-len512_test.npy          8.5 MB
logits_modernbert-patent-len512_val.npy           8.4 MB
logits_modernbert-patent-len8192_test.npy         8.5 MB
logits_modernbert-patent-len8192_val.npy          8.4 MB
modernbert-patent-len512-asl_test_metrics.json      0.0 MB
modernbert-patent-len512-bce_test_metrics.json      0.0 MB
modernbert-patent-len512-zlpr_test_metrics.json      0.0 MB
